# Prefect workflow for running the s3l0 eopf processor

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-643

See the associated:

  * Python module: [s3l0_demo_processor.py](./s3l0_demo_processor.py)
  * YAML file: [s3l0_demo_processor.yaml](./s3l0_demo_processor.yaml)

## 1. Initialisation

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [1]:
import sys
sys.path.insert(0, "/home/jgaucher/projects/rspy/github/rs-demo/notebooks")
import resources.test_localhost

In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
USE_DPR_MOCKUP = True
if os.getenv("RSPY_LOCAL_MODE") == "1" and USE_DPR_MOCKUP:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_ADDRESS"]
    os.environ["DASK_GATEWAY_EOPF_PUBLIC"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_PUBLIC"]

init_demo()
init_dask_cluster_eopf(scale=2, use_mockup = USE_DPR_MOCKUP)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf-mockup': http://dask-eopf-mockup:8000 ...
image = b058ae5ff9e84ed79539761c060754af
Get existing dask cluster: 'b058ae5ff9e84ed79539761c060754af'
Dask dashboard for 'dask-eopf-mockup': http://localhost:8703/clusters/b058ae5ff9e84ed79539761c060754af/status
Dask workers for 'dask-eopf-mockup' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
image = 1c38223b929e4592a429a01fb610aa76
Get existing dask cluster: '1c38223b929e4592a429a01fb610aa76'
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/1c38223b929e4592a429a01fb610aa76/status
Dask workers for 'dask-staging' are up: 2/2


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| numpy       | 2.2.5    | 2.2.4     | 2.2.4    |
| tornado     | 6.3.3    | 6.4.2     | 6.4.2    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 2.2.4     | 2.2.4   |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module

In [3]:
# Create a test collection
TEST_COLLECTION_NAME = "RSPY_643_TEST_COLLECTION"
collection = create_test_collection(TEST_COLLECTION_NAME)

# Check the catalog for RSPY_643_TEST_COLLECTION
items = catalog_client.get_items(TEST_COLLECTION_NAME)
assert not list(items)

#CADIP_SESSION_FILTER = "id=S3A_20250109134406046340" # Session id "platform='sentinel-1a'" "id=S1A_20200105072204051312" S3A_20250109134406046340 | S1A_20200105072204051312
CADIP_SESSION_FILTER ="id=S1A_20200105072204051312"
AUXIP_CQL2_FILTER = {
    "filter": {
        "op": "and",
        "args": [
            {
                "op": "=",
                "args": [
                    {
                        "property": "product:type"
                    },
                    "AX___OSF_AX"
                ]
            },
            {
                "op": "=",
                "args": [
                    {
                        "property": "published"
                    },
                    "2016-01-01T00:00:00.000Z/2016-12-31T23:59:59.999Z"
                ]
            }
        ]
    },
    "sortby": [
        {
            "field": "start_datetime",
            "direction": "desc"
        }
    ],
    "limit": 10
}


15:37:24.255 [INFO] (rs_client.rs_client) Retrieving all items from collection 'jgaucher:RSPY_643_TEST_COLLECTION'.


In [4]:
# Other imports
import getpass
import os
import os.path as osp
from rs_common import prefect_utils
from rs_common.prefect_utils import *

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_SHARE_BUCKET.bucket_name,
    PREFECT_SHARE_BUCKET.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

flow_parameters = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_l0_demo_payload_dpr_mockup_template.yaml",
    "output_data_dir": f"{s3_output}/s3",
    "owner_id": OWNER_ID,
    "collection_name": TEST_COLLECTION_NAME,
    "cadip_stac_filter": CADIP_SESSION_FILTER, 
    "auxip_cql2_filter": AUXIP_CQL2_FILTER,
    "staging_timeout": 120,
    "use_dpr_mockup": USE_DPR_MOCKUP,
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

15:37:24.479 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

15:37:24.481 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

15:37:24.482 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_3A.yaml'.

15:37:24.483 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_dpr_mockup.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'.

15:37:24.520 | INFO    | prefect.S3Bucket - Uploaded 4 files from 'l0/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'

In [5]:
# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name
if cluster_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_SHARE_BUCKET.bucket_name}/{PREFECT_SHARE_BUCKET.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_SHARE_BUCKET.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_SHARE_BUCKET.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_SHARE_BUCKET.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./s3l0_demo_processor.yaml"

14:18:02.586 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| numpy       | 2.2.5    | 2.2.4     | 2.2.4    |
| tornado     | 6.3.3    | 6.4.2     | 6.4.2    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
14:18:02.805 | WARNING | prefect.utilities.templating - Value for placeholder 'JUPYTERHUB_USER' not found in provided values. Please ensure that the placeholder is spelled correctly and that the corresponding value is provided.
14:18:02.806 | WARNIN

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 's3l0-demo-processor/sprint22-s3l0-demo-processor' successfully   │
│ created with id '91344a95-034a-4abc-b87a-e0d11e17d17c'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/91344a95-034a-4abc-b87a-e0d11e17d17c


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
's3l0-demo-processor/sprint22-s3l0-demo-processor'



In [ ]:
deploy_name = "s3l0-demo-processor/sprint22-s3l0-demo-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 's3l0-demo-processor/sprint22-s3l0-demo-processor'


## 3. Run Prefect flow

In [ ]:
output_data_dir = flow_parameters["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3'


In [ ]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
's3l0-demo-processor/sprint22-s3l0-demo-processor'...
Created flow run 'blue-pronghorn'.
└── UUID: 0ff5b246-1fb0-4edc-a822-8a4b0ac021e8
└── Parameters: {'input_config_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/config', 'payload_file': 's3/s3_l0_demo_payload_dpr_mockup_template.yaml', 'output_data_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3', 'owner_id': 'jgaucher', 'collection_name': 'RSPY_643_TEST_COLLECTION', 'cadip_stac_filter': 'id=S1A_20200105072204051312', 'auxip_cql2_filter': {'filter': {'op': 'and', 'args': [{'op': '=', 'args': [{'property': 'product:type'}, 'AX___OSF_AX']}, {'op': '=', 'args': [{'property': 'published'}, '2016-01-01T00:00:00.000Z/2016-12-31T23:59:59.999Z']}]}, 'sortby': [{'field': 'start_datetime', 'direction': 'desc'}], 'limit': 10}, 'staging_timeout': 120, 'use_dpr_mockup': True}
└── Job Variables: {}
└── Scheduled start time: 2025-05-20 14:18:11 UTC (now)
└── URL: http://pr

14:18:12.266 | INFO    | prefect - Flow run is in state 'Pending'
14:18:14.469 | INFO    | prefect - Flow run is in state 'Crashed'


Flow run finished in state 'Crashed'.


CalledProcessError: Command 'b'# Trigger a run for this flow from the command line\nprefect deployment run "$1" --params "$2" --watch\n'' returned non-zero exit status 1.

In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)
eopf_prod_ids = ["S03MWRL0__20221101T092439_6037_A307_T677", "S03OLCL0__20210629T044945_0119_A247_T219"]
for id in eopf_prod_ids:
    assert catalog_client.get_item(TEST_COLLECTION_NAME, id) 
   

## 6. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [6]:
from importlib import reload
debug_flow = True

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *
    os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
    os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name

In [7]:
if debug_flow:
    import s3l0_demo_processor
    reload(s3l0_demo_processor)
    results = s3l0_demo_processor.s3l0_demo_processor(**flow_parameters)
    display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| numpy       | 2.2.5    | 2.2.4     | 2.2.4    |
| tornado     | 6.3.3    | 6.4.2     | 6.4.2    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| numpy       | 2.2.5    | 2.2.4     |

15:37:36.682 | INFO    | Flow run 'daring-seal' - Beginning flow run 'daring-seal' for flow 's3l0-demo-processor'

15:37:36.685 | INFO    | Flow run 'daring-seal' - View at http://prefect-server:4200/runs/flow-run/a7311484-b946-4c81-a2ec-601dae6d1192

15:37:36.949 | INFO    | Flow run 'daring-seal' - For s3_l0_processor found module: l0.s3.s3_l0_processor and processing_unit: S3L0Processor

15:37:36.986 | INFO    | Task run 'cadip-search-222' - Start cadip search

...

15:37:40.818 | ERROR   | Task run 'job-staging-monitor-de0' - Finished in state Failed('Task run encountered an exception FileNotFoundError: The following file path was not found: /opt/conda/lib/python3.11/site-packages/rs_client/ogcapi/../config/staging_templates/yaml/staging_openapi_schema.yaml')

15:37:40.820 | ERROR   | Task run 'job-staging-monitor-b65' - Finished in state Failed('Task run encountered an exception FileNotFoundError: The following file path was not found: /opt/conda/lib/python3.11/site-packages/rs_client/ogcapi/../config/staging_templates/yaml/staging_openapi_schema.yaml')

15:37:40.827 | ERROR   | Flow run 'daring-seal' - Encountered exception during execution: FileNotFoundError('The following file path was not found: /opt/conda/lib/python3.11/site-packages/rs_client/ogcapi/../config/staging_templates/yaml/staging_openapi_schema.yaml')
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/prefect/flow_engine.py", line 765, in run_context
    yield self
  File "/opt/conda/lib/python3.11/site-packages/prefect/flow_engine.py", line 1373, in run_flow_sync
    engine.call_flow_fn()
  File "/opt/conda/lib/python3.11/site-packages/prefect/flow_engine.py", line 785, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/utilities/callables.py", line 208, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/notebooks/sprints/sprint22/s3l0_demo_processor.py", line 219, in s3l0_demo_processor
    staging_cadip_res = cadip_job_staging_monitor_task.result()
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/futures.py", line 223, in result
    _result = self._final_state.result(
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/_internal/compatibility/deprecated.py", line 183, in wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/client/schemas/objects.py", line 320, in result
    return get_state_result(
           ^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/_internal/compatibility/deprecated.py", line 183, in wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/states.py", line 109, in get_state_result
    return _get_state_result(
           ^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/utilities/asyncutils.py", line 351, in coroutine_wrapper
    return run_coro_as_sync(ctx_call())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/utilities/asyncutils.py", line 207, in run_coro_as_sync
    return call.result()
           ^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/_internal/concurrency/calls.py", line 329, in result
    return self.future.result(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/_internal/concurrency/calls.py", line 192, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/concurrent/futures/_base.py", line 401, in __get_result
    raise self._exception
  File "/opt/conda/lib/python3.11/site-packages/prefect/_internal/concurrency/calls.py", line 402, in _run_async
    result = await coro
             ^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/utilities/asyncutils.py", line 188, in coroutine_wrapper
    return await task
           ^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/utilities/asyncutils.py", line 341, in ctx_call
    result = await async_fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/prefect/states.py", line 182, in _get_state_result
    raise await get_state_exception(state)
  File "/opt/conda/lib/python3.11/site-packages/prefect/task_engine.py", line 806, in run_context
    yield self
  File "/opt/conda/lib/python3.11/site-packages/prefect/task_engine.py", line 1388, in run_task_sync
    engine.call_task_fn(txn)
  File "/opt/conda/lib/python3.11/site-packages/prefect/task_engine.py", line 829, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

15:37:40.864 | ERROR   | Flow run 'daring-seal' - Finished in state Failed('Flow run encountered an exception: FileNotFoundError: The following file path was not found: /opt/conda/lib/python3.11/site-packages/rs_client/ogcapi/../config/staging_templates/yaml/staging_openapi_schema.yaml')

FileNotFoundError: The following file path was not found: /opt/conda/lib/python3.11/site-packages/rs_client/ogcapi/../config/staging_templates/yaml/staging_openapi_schema.yaml